In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
!pip install mlflow dagshub -q

import mlflow
import mlflow.sklearn
import dagshub
from kaggle_secrets import UserSecretsClient
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import roc_auc_score

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("ML_assn1")
dagshub_username = user_secrets.get_secret("username")

os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

repo_owner = dagshub_username
repo_name  = "ml_assn2"

mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
mlflow.set_registry_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
dagshub.init(repo_name=repo_name, repo_owner=repo_owner, mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7930128b-428f-4793-b8f8-39be321bacb6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=9ab453adbe4c62d5a9e0bdc9574223466fc5513f4baad3d80b3a67e79681a015




Accessing as lkuch23

Initialized MLflow to track repo "lkuch23/ml_assn2"

Repository lkuch23/ml_assn2 initialized!

In [5]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

# Cleaning

In [6]:
missing_ratio = train.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.85].index
train.drop(cols_to_drop, axis=1, inplace=True)
test.drop(columns=[c for c in cols_to_drop if c in test.columns], inplace=True)

In [7]:
for col in train.columns:
    if col == 'isFraud':
        continue
    if col in test.columns:
        if train[col].dtype == 'object':
            train[col].fillna('missing', inplace=True)
            test[col].fillna('missing', inplace=True)
        else:
            median = train[col].median()
            train[col].fillna(median, inplace=True)
            test[col].fillna(median, inplace=True)

/tmp/ipykernel_57/2179522781.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col].fillna(median, inplace=True)
/tmp/ipykernel_57/2179522781.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.m

# Feature Engineering

In [8]:
train['hour'] = (train['TransactionDT'] // 3600) % 24
test['hour'] = (test['TransactionDT'] // 3600) % 24
train['day_of_week'] = (train['TransactionDT'] // (3600*24)) % 7
test['day_of_week'] = (test['TransactionDT'] // (3600*24)) % 7
train['amt_log'] = np.log1p(train['TransactionAmt'])
test['amt_log'] = np.log1p(test['TransactionAmt'])
train['amt_cents'] = train['TransactionAmt'] - train['TransactionAmt'].astype(int)
test['amt_cents'] = test['TransactionAmt'] - test['TransactionAmt'].astype(int)
train['amt_per_hour'] = train['TransactionAmt'] / (train['hour'] + 1)
test['amt_per_hour'] = test['TransactionAmt'] / (test['hour'] + 1)

In [9]:
common_cols = list(set(train.columns) & set(test.columns))
cat_cols = [col for col in common_cols if train[col].dtype == 'object']

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]]).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Feature Selection

In [10]:
target = 'isFraud'
ID = 'TransactionID'

common_cols = set(train.columns) & set(test.columns)
candidate_features = sorted([col for col in common_cols if col not in [target, ID]])

X_all = train[candidate_features].apply(pd.to_numeric, errors='coerce').fillna(0)
y = train[target]

vt = VarianceThreshold(threshold=0.01)
vt.fit(X_all)
X_all = pd.DataFrame(vt.transform(X_all), columns=np.array(candidate_features)[vt.get_support()])

sample_idx = np.random.choice(len(X_all), size=min(50000, len(X_all)), replace=False)
mi_scores = mutual_info_classif(X_all.iloc[sample_idx], y.iloc[sample_idx], random_state=42)
mi_df = pd.DataFrame({'feature': X_all.columns, 'mi': mi_scores}).sort_values('mi', ascending=False)

TOP_K = 150
selected_features = mi_df.head(TOP_K)['feature'].tolist()

# Training

In [11]:
X = train[selected_features].apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_raw = test[selected_features].apply(pd.to_numeric, errors='coerce').fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test_raw)

In [12]:
mlflow.set_experiment("IEEE_Logistic_Regression")

<Experiment: artifact_location='mlflow-artifacts:/881adbd8eb454ad0baec6832dc7368bb', creation_time=1777890488147, experiment_id='1', last_update_time=1777890488147, lifecycle_stage='active', name='IEEE_Logistic_Regression', tags={}, trace_location=None, workspace='default'>

In [13]:
params = {
    'C': 0.5,
    'class_weight': "balanced",
    'solver': "lbfgs",
    'max_iter': 1000,
    'random_state': 42,
    'n_jobs': -1,
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X_scaled))
test_preds = np.zeros(len(X_test_scaled))
fold_aucs  = []

with mlflow.start_run():
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        X_trn, X_val = X_scaled[trn_idx], X_scaled[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        model = LogisticRegression(**params)
        model.fit(X_trn, y_trn)
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test_scaled)[:, 1] / N_FOLDS
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_aucs.append(fold_auc)
        
    oof_auc = roc_auc_score(y, oof_preds)
    mlflow.log_metric("AUC", oof_auc)
    mlflow.sklearn.log_model(model, "model")

2026/05/05 21:05:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 21:05:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run defiant-foal-275 at: https://dagshub.com/lkuch23/ml_assn2.mlflow/#/experiments/1/runs/20a590b6e4cd4b58aa4c8cda5128c58f
🧪 View experiment at: https://dagshub.com/lkuch23/ml_assn2.mlflow/#/experiments/1
